In [1]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# --- Load Data ---
try:
    df_filled = pd.read_csv("DC_Master_Data_Filled.csv", parse_dates=['Date'], index_col='Date')
    # Ensure the frequency is set correctly
    df_filled = df_filled.asfreq('QS-JAN')
    print("Data loaded successfully.")

    # --- Stationarity Test Function ---
    def test_stationarity(timeseries, var_name, regression_kpss='c'):
        """Performs ADF and KPSS tests and prints results."""
        print(f"\n--- Stationarity Tests for {var_name} ---")
        stationary = False # Flag to track stationarity

        # Drop NaNs that might result from differencing
        timeseries = timeseries.dropna()
        if timeseries.empty:
            print("Series is empty after dropping NaNs, cannot test.")
            return False # Indicate series couldn't be tested

        # --- ADF Test ---
        try:
            # Null Hypothesis: Series is non-stationary (has a unit root)
            adf_result = adfuller(timeseries, autolag='AIC')
            adf_stat = adf_result[0]
            adf_pvalue = adf_result[1]
            print(f"ADF Statistic: {adf_stat:.3f}")
            print(f"ADF p-value: {adf_pvalue:.3f}")
            adf_stationary = adf_pvalue < 0.05
            if adf_stationary:
                print("ADF Result: Likely Stationary (reject H0)")
            else:
                print("ADF Result: Likely Non-Stationary (fail to reject H0)")
        except Exception as e:
            print(f"ADF Test Failed: {e}")
            adf_stationary = False # Treat failure as non-stationary indication

        # --- KPSS Test ---
        try:
            # Null Hypothesis: Series is stationary (around a constant level or trend)
            kpss_result = kpss(timeseries, regression=regression_kpss, nlags="auto")
            kpss_stat = kpss_result[0]
            kpss_pvalue = kpss_result[1]
            print(f"\nKPSS Statistic ({regression_kpss}): {kpss_stat:.3f}")
            print(f"KPSS p-value ({regression_kpss}): {kpss_pvalue:.3f}")
            # If p < critical value (e.g., 0.05), reject H0 (suggests non-stationarity)
            kpss_stationary = kpss_pvalue >= 0.05
            if kpss_stationary:
                 print(f"KPSS Result ({regression_kpss}): Likely Stationary (fail to reject H0)")
            else:
                 print(f"KPSS Result ({regression_kpss}): Likely Non-Stationary (reject H0)")
        except Exception as e:
            print(f"KPSS Test Failed ({regression_kpss}): {e}")
            kpss_stationary = False # Treat failure cautiously

        # Combined Interpretation
        if adf_stationary and kpss_stationary:
             print("Combined: Likely Stationary")
             stationary = True
        elif not adf_stationary and not kpss_stationary:
             print("Combined: Likely Non-Stationary (Unit Root)")
             stationary = False
        else: # Conflicting results
             print("Combined: Results are conflicting or inconclusive. Examine plots.")
             # Be conservative: if either test suggests non-stationarity, assume non-stationary
             stationary = False

        return stationary # Return the stationarity flag

    # --- Test Variables ---
    integration_orders = {}
    numeric_cols = df_filled.select_dtypes(include=['number']).columns

    for col in numeric_cols:
        print(f"\n{'='*15} Testing Variable: {col} {'='*15}")

        # Test original series (level) - check for stationarity around constant ('c')
        # If it looks strongly trended, one might also check 'ct', but let's start with 'c'
        is_i0 = test_stationarity(df_filled[col], f"{col} (Original)", regression_kpss='c')
        if is_i0:
            integration_orders[col] = 'I(0)'
            print(f"Conclusion for {col}: Likely I(0)")
            print("-" * 50)
            continue # Skip differencing if already stationary

        # Test first difference
        diff1_series = df_filled[col].diff()
        is_i1 = test_stationarity(diff1_series, f"{col} (1st Difference)", regression_kpss='c')
        if is_i1:
            integration_orders[col] = 'I(1)'
            print(f"Conclusion for {col}: Likely I(1)")
            print("-" * 50)
            continue # Stop if first difference is stationary

        # Test second difference (only if first difference wasn't stationary)
        print(f"\nFirst difference of {col} appears non-stationary, testing 2nd difference...")
        diff2_series = df_filled[col].diff().diff()
        is_i2 = test_stationarity(diff2_series, f"{col} (2nd Difference)", regression_kpss='c')
        if is_i2:
            integration_orders[col] = 'I(2)'
            print(f"Conclusion for {col}: Likely I(2)")
        else:
            integration_orders[col] = 'I(>2) or Complex'
            print(f"Conclusion for {col}: Requires more than 2 differences or has complex structure.")
        print("-" * 50)

    # --- Summarize Integration Orders ---
    print("\n\n--- Summary of Integration Orders ---")
    for var, order in integration_orders.items():
        print(f"{var}: {order}")

except FileNotFoundError:
    print("Error: The file 'DC_Master_Data_Filled.csv' was not found.")
except Exception as e:
    print(f"An error occurred during analysis: {e}")

Data loaded successfully.

=============== Testing Variable: House_Index ===============

--- Stationarity Tests for House_Index (Original) ---
ADF Statistic: -0.199
ADF p-value: 0.939
ADF Result: Likely Non-Stationary (fail to reject H0)

KPSS Statistic (c): 1.835
KPSS p-value (c): 0.010
KPSS Result (c): Likely Non-Stationary (reject H0)
Combined: Likely Non-Stationary (Unit Root)

--- Stationarity Tests for House_Index (1st Difference) ---
ADF Statistic: -3.047
ADF p-value: 0.031
ADF Result: Likely Stationary (reject H0)

KPSS Statistic (c): 0.231
KPSS p-value (c): 0.100
KPSS Result (c): Likely Stationary (fail to reject H0)
Combined: Likely Stationary
Conclusion for House_Index: Likely I(1)
--------------------------------------------------

=============== Testing Variable: CPI ===============

--- Stationarity Tests for CPI (Original) ---
ADF Statistic: 1.356
ADF p-value: 0.997
ADF Result: Likely Non-Stationary (fail to reject H0)

KPSS Statistic (c): 2.090
KPSS p-value (c): 0.010

In [2]:
# --- Load Data ---
try:
    df_filled = pd.read_csv("DC_Master_Data_Filled.csv", parse_dates=['Date'], index_col='Date')
    # Ensure the frequency is set correctly
    df_filled = df_filled.asfreq('QS-JAN')
    print("Data loaded successfully.")

    # --- Calculate 3rd Difference for GDP ---
    gdp_diff3 = df_filled['GDP'].diff().diff().diff().dropna()
    print(f"\nCalculating 3rd difference for GDP (length: {len(gdp_diff3)} points)")

    # --- Test Stationarity of 3rd Difference ---
    if not gdp_diff3.empty:
        print("\n--- Stationarity Tests for GDP (3rd Difference) ---")
        stationary_3rd_diff = False

        # ADF Test
        try:
            adf_result_3 = adfuller(gdp_diff3, autolag='AIC')
            adf_stat_3 = adf_result_3[0]
            adf_pvalue_3 = adf_result_3[1]
            print(f"ADF Statistic: {adf_stat_3:.3f}")
            print(f"ADF p-value: {adf_pvalue_3:.3f}")
            adf_stationary_3 = adf_pvalue_3 < 0.05
            if adf_stationary_3:
                print("ADF Result: Likely Stationary (reject H0)")
            else:
                print("ADF Result: Likely Non-Stationary (fail to reject H0)")
        except Exception as e:
            print(f"ADF Test Failed: {e}")
            adf_stationary_3 = False

        # KPSS Test
        try:
            kpss_result_3 = kpss(gdp_diff3, regression='c', nlags="auto")
            kpss_stat_3 = kpss_result_3[0]
            kpss_pvalue_3 = kpss_result_3[1]
            print(f"\nKPSS Statistic (c): {kpss_stat_3:.3f}")
            print(f"KPSS p-value (c): {kpss_pvalue_3:.3f}")
            kpss_stationary_3 = kpss_pvalue_3 >= 0.05
            if kpss_stationary_3:
                 print(f"KPSS Result (c): Likely Stationary (fail to reject H0)")
            else:
                 print(f"KPSS Result (c): Likely Non-Stationary (reject H0)")
        except Exception as e:
            print(f"KPSS Test Failed (c): {e}")
            kpss_stationary_3 = False

        # Combined Interpretation
        if adf_stationary_3 and kpss_stationary_3:
             print("Combined: Likely Stationary (I(3))")
             stationary_3rd_diff = True
        elif not adf_stationary_3 and not kpss_stationary_3:
             print("Combined: Likely Non-Stationary (Unit Root even after 3 diffs)")
             stationary_3rd_diff = False
        else: # Conflicting results
             print("Combined: Results are conflicting or inconclusive.")
             stationary_3rd_diff = False # Treat conflict as non-stationary

        print(f"\nConclusion for GDP after 3 differences: {'Likely Stationary I(3)' if stationary_3rd_diff else 'Still Not Clearly Stationary'}")

    else:
        print("Could not perform test on 3rd difference as the series became empty.")

except FileNotFoundError:
    print("Error: The file 'DC_Master_Data_Filled.csv' was not found.")
except Exception as e:
    print(f"An error occurred during analysis: {e}")

Data loaded successfully.

Calculating 3rd difference for GDP (length: 137 points)

--- Stationarity Tests for GDP (3rd Difference) ---
ADF Statistic: -6.563
ADF p-value: 0.000
ADF Result: Likely Stationary (reject H0)

KPSS Statistic (c): 0.144
KPSS p-value (c): 0.100
KPSS Result (c): Likely Stationary (fail to reject H0)
Combined: Likely Stationary (I(3))

Conclusion for GDP after 3 differences: Likely Stationary I(3)


In [3]:
import pandas as pd
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# --- Step 1: Load Data ---
try:
    df_filled = pd.read_csv("DC_Master_Data_Filled.csv", parse_dates=['Date'], index_col='Date')
    # Ensure the frequency is set correctly
    df_filled = df_filled.asfreq('QS-JAN')
    print("Data loaded successfully.")

    # --- Step 2: Select ALL I(1) Variables for Testing ---
    # Based on previous stationarity test results
    vars_to_test_all = [
        'House_Index',
        'CPI',
        'Poverty_Rate', # Added
        'Median_Household_Income',
        'Interest_Rate', # Added
        'Mortgage_Rate'
        ]
    df_test_all = df_filled[vars_to_test_all].copy()
    print(f"\nVariables selected for cointegration test: {vars_to_test_all}")
    print(f"Shape of data for testing: {df_test_all.shape}")

    # --- Step 3: Perform Johansen Cointegration Test ---
    # Using same parameters as before: det_order=0, k_ar_diff=1
    det_order = 0
    k_ar_diff = 1
    print(f"\nRunning Johansen test with det_order={det_order} and k_ar_diff={k_ar_diff} on all {len(vars_to_test_all)} I(1) variables...")

    johansen_result_all = coint_johansen(df_test_all, det_order=det_order, k_ar_diff=k_ar_diff)

    # --- Step 4: Interpret and Print Results ---
    print("\n--- Johansen Cointegration Test Results (All I(1) Variables) ---")

    def print_johansen_interpretation(stat, crit_vals, stat_name, num_vars):
        """Helper function to print interpretation"""
        print(f"\n{stat_name} Statistics:")
        print(" r | Statistic | 90% Crit Value | 95% Crit Value | 99% Crit Value | Reject H0 (95%)?")
        print("---|-----------|----------------|----------------|----------------|-----------------")
        for i in range(num_vars):
            r = i # Null hypothesis is r <= i (for trace) or r = i (for max_eig)
            s = stat[i]
            cv90 = crit_vals[i, 0]
            cv95 = crit_vals[i, 1]
            cv99 = crit_vals[i, 2]
            reject_95 = "Yes" if s > cv95 else "No"
            print(f" {r} |  {s:8.3f} |     {cv90:8.3f} |     {cv95:8.3f} |     {cv99:8.3f} | {reject_95}")

    num_tested_vars = len(vars_to_test_all)

    # Trace Statistic Interpretation
    print("Trace Statistic Test:")
    print("Tests the null hypothesis that the number of cointegrating vectors is r <= k.")
    print("Start from r=0. If Stat > Critical Value, reject H0 and move to r=1, etc.")
    print_johansen_interpretation(johansen_result_all.lr1, johansen_result_all.cvt, "Trace", num_tested_vars)
    trace_rank = 0
    for i in range(num_tested_vars):
        if johansen_result_all.lr1[i] > johansen_result_all.cvt[i, 1]: # Check against 95% critical value
             trace_rank += 1
        else:
            break

    # Max Eigenvalue Statistic Interpretation
    print("\nMaximum Eigenvalue Statistic Test:")
    print("Tests the null hypothesis that the number of cointegrating vectors is r = k vs r = k+1.")
    print("Start from r=0. If Stat > Critical Value, reject H0 and move to r=1, etc.")
    print_johansen_interpretation(johansen_result_all.lr2, johansen_result_all.cvm, "Max Eigenvalue", num_tested_vars)
    max_eig_rank = 0
    for i in range(num_tested_vars):
        if johansen_result_all.lr2[i] > johansen_result_all.cvm[i, 1]: # Check against 95% critical value
             max_eig_rank += 1
        else:
            break


    # --- Conclusion ---
    print("\n--- Cointegration Rank Conclusion (All I(1) Variables) ---")
    print(f"Trace test indicates {trace_rank} cointegrating vector(s) at 95% significance.")
    print(f"Max Eigenvalue test indicates {max_eig_rank} cointegrating vector(s) at 95% significance.")

    # Check for agreement
    if trace_rank == max_eig_rank:
        print(f"\nBoth tests suggest a cointegrating rank r = {trace_rank}.")
        if trace_rank > 0:
            print("This suggests a stable long-run relationship exists among the variables.")
            print("A Vector Error Correction Model (VECM) is likely appropriate.")
        else:
            print("This suggests no stable long-run relationship (no cointegration).")
            print("A VAR model on the differenced data is likely appropriate.")
    else:
        print("\nThe tests suggest different ranks. This can happen and requires judgment.")
        print("Consider the economic theory, visual inspection of series, or trying different lag orders (k_ar_diff).")
        print(f"Based on Trace: r={trace_rank}. Based on Max Eigenvalue: r={max_eig_rank}.")


except FileNotFoundError:
    print("Error: The file 'DC_Master_Data_Filled.csv' was not found.")
except Exception as e:
    print(f"An error occurred during analysis: {e}")

Data loaded successfully.

Variables selected for cointegration test: ['House_Index', 'CPI', 'Poverty_Rate', 'Median_Household_Income', 'Interest_Rate', 'Mortgage_Rate']
Shape of data for testing: (140, 6)

Running Johansen test with det_order=0 and k_ar_diff=1 on all 6 I(1) variables...

--- Johansen Cointegration Test Results (All I(1) Variables) ---
Trace Statistic Test:
Tests the null hypothesis that the number of cointegrating vectors is r <= k.
Start from r=0. If Stat > Critical Value, reject H0 and move to r=1, etc.

Trace Statistics:
 r | Statistic | 90% Crit Value | 95% Crit Value | 99% Crit Value | Reject H0 (95%)?
---|-----------|----------------|----------------|----------------|-----------------
 0 |    85.521 |       91.109 |       95.754 |      104.964 | No
 1 |    54.253 |       65.820 |       69.819 |       77.820 | No
 2 |    35.042 |       44.493 |       47.855 |       54.681 | No
 3 |    19.835 |       27.067 |       29.796 |       35.463 | No
 4 |     7.529 |      

In [4]:
import pandas as pd
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# --- Step 1: Load Data ---
try:
    df_filled = pd.read_csv("DC_Master_Data_Filled.csv", parse_dates=['Date'], index_col='Date')
    # Ensure the frequency is set correctly
    df_filled = df_filled.asfreq('QS-JAN')
    print("Data loaded successfully.")

    # --- Step 2: Apply Differencing based on Integration Orders ---
    df_stationary = pd.DataFrame(index=df_filled.index) # Initialize new DataFrame

    # I(0) Variable
    df_stationary['Unemployment_Rate'] = df_filled['Unemployment_Rate'] # Keep original levels

    # I(1) Variables
    i1_vars = ['House_Index', 'CPI', 'Poverty_Rate', 'Median_Household_Income', 'Interest_Rate', 'Mortgage_Rate']
    for var in i1_vars:
        df_stationary[f'{var}_diff1'] = df_filled[var].diff(1)

    # I(2) Variable
    df_stationary['Population_diff2'] = df_filled['Population'].diff(2)

    # I(3) Variable
    df_stationary['GDP_diff3'] = df_filled['GDP'].diff(3)

    # --- Step 3: Drop NaNs introduced by differencing ---
    # The highest order of differencing is 3, so the first 3 rows will have NaNs
    df_stationary.dropna(inplace=True)

    # --- Step 4: Display Results ---
    print(f"\nStationary DataFrame created after differencing and dropping NaNs.")
    print(f"Shape of the stationary DataFrame: {df_stationary.shape}")

    print("\n--- First 5 Rows of Stationary Data ---")
    print(df_stationary.head())

    print("\n--- Last 5 Rows of Stationary Data ---")
    print(df_stationary.tail())

except FileNotFoundError:
    print("Error: The file 'DC_Master_Data_Filled.csv' was not found.")
except Exception as e:
    print(f"An error occurred during analysis: {e}")

Data loaded successfully.

Stationary DataFrame created after differencing and dropping NaNs.
Shape of the stationary DataFrame: (137, 9)

--- First 5 Rows of Stationary Data ---
            Unemployment_Rate  House_Index_diff1  CPI_diff1  \
Date                                                          
1990-10-01                4.9              -1.34       2.50   
1991-01-01                5.3               0.54       0.90   
1991-04-01                5.2               0.15       1.00   
1991-07-01                6.1              -0.81       0.80   
1991-10-01                6.2               1.60       2.05   

            Poverty_Rate_diff1  Median_Household_Income_diff1  \
Date                                                            
1990-10-01               0.175                          770.0   
1991-01-01               0.175                          770.0   
1991-04-01               0.150                         -200.0   
1991-07-01               0.150                        

In [5]:
import pandas as pd
from statsmodels.tsa.api import VAR
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# --- Step 1: Load Data ---
# (Assuming df_filled is already loaded from previous steps, otherwise reload)
try:
    df_filled = pd.read_csv("DC_Master_Data_Filled.csv", parse_dates=['Date'], index_col='Date')
    df_filled = df_filled.asfreq('QS-JAN')

    # --- Step 2: Apply Differencing (as done previously) ---
    df_stationary = pd.DataFrame(index=df_filled.index)

    # I(0) Variable
    df_stationary['Unemployment_Rate'] = df_filled['Unemployment_Rate']

    # I(1) Variables
    i1_vars = ['House_Index', 'CPI', 'Poverty_Rate', 'Median_Household_Income', 'Interest_Rate', 'Mortgage_Rate']
    for var in i1_vars:
        df_stationary[f'{var}_diff1'] = df_filled[var].diff(1)

    # I(2) Variable
    df_stationary['Population_diff2'] = df_filled['Population'].diff(2)

    # I(3) Variable
    df_stationary['GDP_diff3'] = df_filled['GDP'].diff(3)

    # Drop NaNs
    df_stationary.dropna(inplace=True)

    if df_stationary.empty:
        print("Stationary DataFrame is empty after differencing and dropping NaNs.")
    else:
        print("Stationary data prepared successfully.")
        print(f"Shape of stationary data for VAR: {df_stationary.shape}")

        # --- Step 3: VAR Lag Order Selection ---
        print("\nSelecting optimal VAR lag order...")
        # Instantiate the VAR model with the stationary data
        model_var = VAR(df_stationary)

        # Select the lag order using information criteria
        # Check up to 12 lags (3 years of quarterly data) - adjust if needed
        maxlags = 12
        try:
          lag_selection_results = model_var.select_order(maxlags=maxlags)
          print(lag_selection_results.summary())
        except Exception as e:
            print(f"Error during lag order selection: {e}")
            print("This might occur if maxlags is too high for the dataset size after differencing.")
            print("Consider reducing maxlags.")


except FileNotFoundError:
    print("Error: The file 'DC_Master_Data_Filled.csv' was not found.")
except Exception as e:
    print(f"An error occurred during analysis: {e}")

Stationary data prepared successfully.
Shape of stationary data for VAR: (137, 9)

Selecting optimal VAR lag order...
 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0        28.96       29.16   3.768e+12       29.04
1        22.05      24.09*   3.786e+09       22.88
2        21.57       25.44   2.376e+09       23.14
3        21.17       26.87   1.668e+09       23.48
4        20.36       27.89   8.195e+08       23.42
5        19.43       28.80   3.807e+08       23.24
6        19.00       30.20   3.179e+08       23.55
7        18.38       31.41   2.513e+08       23.67
8        17.69       32.55   2.195e+08       23.72
9        16.82       33.52   2.090e+08       23.61
10       15.92       34.46   2.830e+08       23.45
11       13.90       34.26   2.352e+08       22.17
12      6.986*       29.18  5.047e+06*      16.00*
--------------------------------------------------


In [6]:
import pandas as pd
from statsmodels.tsa.api import VAR
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# --- Step 1: Load Data ---
try:
    df_filled = pd.read_csv("DC_Master_Data_Filled.csv", parse_dates=['Date'], index_col='Date')
    # Ensure the frequency is set correctly
    df_filled = df_filled.asfreq('QS-JAN')
    print("Data loaded successfully.")

    # --- Step 2: Prepare Stationary Data ---
    df_stationary = pd.DataFrame(index=df_filled.index) # Initialize new DataFrame

    # I(0) Variable
    df_stationary['Unemployment_Rate'] = df_filled['Unemployment_Rate'] # Keep original levels

    # I(1) Variables
    i1_vars = ['House_Index', 'CPI', 'Poverty_Rate', 'Median_Household_Income', 'Interest_Rate', 'Mortgage_Rate']
    for var in i1_vars:
        df_stationary[f'{var}_diff1'] = df_filled[var].diff(1)

    # I(2) Variable
    df_stationary['Population_diff2'] = df_filled['Population'].diff(2)

    # I(3) Variable
    df_stationary['GDP_diff3'] = df_filled['GDP'].diff(3)

    # Drop NaNs introduced by differencing
    df_stationary.dropna(inplace=True)
    print(f"Stationary data prepared (Shape: {df_stationary.shape}).")

    # --- Step 3: Fit VAR(1) Model ---
    if not df_stationary.empty:
        print("\nFitting VAR model with lag order p=1...")
        # Instantiate the VAR model with the stationary data
        model_var = VAR(df_stationary)

        # Fit the model with lag=1
        var1_results = model_var.fit(1)
        print("VAR(1) model fitting complete.")

        # --- Step 4: Display VAR(1) Summary ---
        print("\n--- VAR(1) Model Summary ---")
        # The summary can be very long with 9 variables, but provides details
        print(var1_results.summary())

    else:
        print("Stationary DataFrame is empty, cannot fit VAR model.")

except FileNotFoundError:
    print("Error: The file 'DC_Master_Data_Filled.csv' was not found.")
except Exception as e:
    print(f"An error occurred during analysis: {e}")

Data loaded successfully.
Stationary data prepared (Shape: (137, 9)).

Fitting VAR model with lag order p=1...
VAR(1) model fitting complete.

--- VAR(1) Model Summary ---
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Wed, 16, Apr, 2025
Time:                     20:57:16
--------------------------------------------------------------------
No. of Equations:         9.00000    BIC:                    23.6786
Nobs:                     136.000    HQIC:                   22.5344
Log likelihood:          -3125.85    FPE:                2.80162e+09
AIC:                      21.7511    Det(Omega_mle):     1.47939e+09
--------------------------------------------------------------------
Results for equation Unemployment_Rate
                                      coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------------------------------
